# ImpactLens: train on Kaggle (free GPU)

Trains the two models on the packed datasets: a Siamese U-Net for change (OSCD) and a plain U-Net for flood water (Sen1Floods11).

**Before you run it** (right-hand *Notebook options* / *Session options* panel):
1. **Accelerator**: GPU (T4 x2 or P100).
2. **Internet**: On (needed to `git clone`; Kaggle asks you to verify your phone number once).
3. **Add data**: your private dataset `impactlens-packed` (the two `.npz` files from `scripts/pack_for_kaggle.py`).
4. Edit `REPO_URL` below to your GitHub repo.

Evaluation rule: checkpoints are chosen on *validation* data only (OSCD: 3 held-out train cities; Sen1Floods11: its `valid` split). The test splits are scored once, at the end.

In [ ]:
# ---- settings you may edit -------------------------------------------------------------
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/impactlens.git"  # <- your repo (public)
BRANCH = "main"
WORK = "/kaggle/working"

## 1. Get the code

In [ ]:
import os
import subprocess

REPO_DIR = f"{WORK}/impactlens"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
print(
    subprocess.run(
        ["git", "-C", REPO_DIR, "log", "--oneline", "-3"], capture_output=True, text=True
    ).stdout
)

Install the package **without dependencies**: Kaggle already ships a CUDA build of PyTorch and numpy,
and we must not let pip replace it. Packed data needs no rasterio, so nothing else is required.
`--ignore-requires-python` because the repo pins Python 3.11 and Kaggle's Python may differ.

In [ ]:
!pip install -q --no-deps --ignore-requires-python -e {REPO_DIR}
import sys

# A fresh editable install is not always visible to an already-running kernel; make sure it is.
sys.path.insert(0, f"{REPO_DIR}/src")
import impactlens

print("impactlens", impactlens.__version__)

## 2. Check the GPU and find the data

In [ ]:
import glob

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Turn on an accelerator in the session options.")

hits = glob.glob("/kaggle/input/**/oscd.npz", recursive=True)
assert hits, "oscd.npz not found: add your impactlens-packed dataset via 'Add data'"
DATA = os.path.dirname(hits[0])
print("data dir:", DATA)
for f in sorted(os.listdir(DATA)):
    print(f"  {f:24} {os.path.getsize(os.path.join(DATA, f)) / 1e6:8.1f} MB")

## 3. Smoke-test the data
Loads every split through the real dataset classes. If a count is off, stop here and re-pack.

In [ ]:
from impactlens.data.oscd import OSCDDataset
from impactlens.data.sen1floods11 import Sen1Floods11Dataset

oscd_npz, s1f_npz = f"{DATA}/oscd.npz", f"{DATA}/sen1floods11.npz"
for split in ("train", "val", "test"):
    print(f"OSCD {split:5}: {len(OSCDDataset(oscd_npz, split)):4} fixed tiles")
print(
    "OSCD train random crops per epoch: e.g.",
    len(OSCDDataset(oscd_npz, "train", crops_per_epoch=500)),
)
for split in ("train", "valid", "test", "bolivia"):
    print(f"Sen1Floods11 {split:8}: {len(Sen1Floods11Dataset(s1f_npz, split)):4} tiles")

item = OSCDDataset(oscd_npz, "train", crops_per_epoch=1, augment=True)[0]
print({k: (tuple(v.shape), round(float(v.mean()), 3)) for k, v in item.items() if k != "city"})

## 4. Train
`train.py` is built in **Phase 3**. Until it exists this cell only prints a reminder.
Once it exists, it trains both models, saves checkpoints + metrics to `/kaggle/working/runs/`,
and exports ONNX. Outputs in `/kaggle/working` are kept when you press **Save Version**.

In [ ]:
import importlib.util

RUNS = f"{WORK}/runs"
if importlib.util.find_spec("impactlens.models.train") is None:
    print("train.py not written yet (Phase 3): data pipeline is ready, stopping here.")
else:
    !python -m impactlens.models.train --task change --data {DATA} --out {RUNS}/change
    !python -m impactlens.models.train --task water  --data {DATA} --out {RUNS}/water

## 5. Collect results
Download from the notebook's **Output** tab (or `kaggle kernels output <user>/<notebook> -p results/`),
then drop the checkpoints/ONNX files into the repo's `models/` folder (gitignored) locally.

In [ ]:
for root, _, files in os.walk(RUNS if os.path.isdir(RUNS) else WORK):
    if "impactlens" in root.split(os.sep) or "/.git" in root:
        continue
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p}")